# ML Study — จาก Warehouse สู่ผลพยากรณ์

Notebook นี้ใช้ **ข้อมูลสังเคราะห์เพื่อการศึกษา** และรันใน Colab ของผู้เปิดเอง ไม่เชื่อม Neon Lab หรือเปลี่ยน Dashboard บนเว็บ

**โจทย์:** ทำนายยอดขายสุทธิรายสัปดาห์แยกกลุ่มสินค้า จากประวัติใน DuckDB Warehouse และแผนการตลาดที่บันทึกก่อนสัปดาห์ที่จะทำนาย

เลือก **Runtime → Run all** แล้วดูผลใต้แต่ละ Cell การรันอาจใช้เวลาหลายนาทีเพราะสร้างข้อมูล ประมวลผล Pipeline และฝึกโมเดลจริง ผลการประเมินอาจไม่ผ่านเกณฑ์ ซึ่งเป็นผลที่ต้องแสดงตามจริง


## 1. เตรียม Code และพื้นที่ทดลอง

In [ ]:
#@title 1 — เตรียมโปรเจกต์ใน Colab
from pathlib import Path
import os, sys, subprocess, json
REPO_URL = "https://github.com/KankaweeK356/tiktok-shop-pipeline-demo"
REPO_REF = "feature/classroom-demo-evidence-foundation"
ROOT = Path("/content/TikTok_Shop_Pipeline_Demo_ML") if Path("/content").is_dir() else Path.cwd()
if not (ROOT / "pipeline" / "ci_ml_history.py").is_file():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError("โฟลเดอร์โปรเจกต์มีไฟล์อยู่แล้ว; เปิด Colab Runtime ใหม่ก่อน")
    subprocess.check_call(["git", "clone", "--depth", "1", "--single-branch",
                           "--branch", REPO_REF, REPO_URL, str(ROOT)])
else:
    current = subprocess.check_output(["git", "-C", str(ROOT),
                                       "branch", "--show-current"], text=True).strip()
    if current != REPO_REF:
        raise RuntimeError("Code ใน Runtime เป็นคนละสาขา; เปิด Runtime ใหม่ก่อน")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT.resolve()))
os.environ.pop("NEON_DEMO_DATABASE_URL", None)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(ROOT / "requirements-ml.txt")])
from IPython.display import display
import pandas as pd
from pipeline.orchestrator import reset, run_batch
from pipeline.generate_classroom_source import build, BATCHES
BASE = ROOT / "output" / "ml_classroom"
SOURCE = BASE / "source"
WORKSPACE = BASE / "main"
build(SOURCE)
reset(WORKSPACE)
print("โค้ด:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("พื้นที่ผลทดลอง:", BASE)
print("Neon Lab ไม่ถูกเชื่อมต่อ")


## 2. ใช้ Source ทั้ง 7 ชุดผ่าน Database → Data Lake → Warehouse

ทุกชุดใช้โครงสร้าง ER เดิมและตรวจยอดก่อนฝึกโมเดล ไม่มี Warehouse ประวัติอีกชุดที่ซ่อนอยู่


In [ ]:
#@title 2 — รัน Source ทุกชุดและตรวจ Warehouse
for batch in BATCHES:
    result = run_batch(WORKSPACE, batch, source_root=SOURCE)
    assert result["status"] == "SUCCESS" and result["functional_fails"] == 0
    print(batch, "PASS", "| รายการขาย", result["paid_sales_rows"])
with __import__("duckdb").connect(str(WORKSPACE / "er2.duckdb"), read_only=True) as db:
    months = db.execute('SELECT Year_Month,Net_Revenue,Contribution_Profit,Profit_Protection_Status FROM View_Shop_Monthly_KPI ORDER BY Year_Month').df()
assert len(months) >= 18
display(months.tail(8))
print("เดือนที่มีใน Warehouse:", len(months))


## 3. สร้าง Feature และฝึกโมเดล

หนึ่งแถวแทน **กลุ่มสินค้า × สัปดาห์เป้าหมาย × วันที่ตัดข้อมูล** ใช้ยอดย้อนหลังและแผนที่ทราบก่อนวันทำนาย แบ่ง Train/Validation/Test ตามเวลา ไม่สุ่มแถวปนกัน และเลือกโมเดลจาก Validation ก่อนเปิดผล Test

**ข้อจำกัด:** Warehouse ประวัติเป็น snapshot สถานะล่าสุด จึงยังพิสูจน์ไม่ได้ว่า Feature ทุกค่ามีอยู่จริง ณ เวลาทำนายในอดีต เรียกผลนี้ว่า retrospective offline study ไม่ใช่ point-in-time verified production model


In [ ]:
#@title 3 — ฝึกและประเมินโมเดลจาก Warehouse
from pipeline.ml_weekly import run_weekly
OUT = BASE / "reports/ml_weekly"
summary = run_weekly(WORKSPACE / "er2.duckdb",
                     SOURCE / "ml_marketing_plan.csv", OUT)
assert summary["test_not_used_for_model_selection"]
assert summary["fidelity"] == "RETROSPECTIVE_LATEST_DWH"
features = pd.read_parquet(OUT / "weekly_training_dataset.parquet")
print("Feature rows:", len(features), "| สัปดาห์:", summary["weeks"],
      "| กลุ่มสินค้า:", summary["groups"])
print("Validation folds:", ", ".join(summary["validation_folds"]))
print("Locked Test:", " ถึง ".join(summary["final_test_window"]))
display(features[["as_of_date", "forecast_week_start", "product_group_id",
                  "revenue_last_week", "planned_ads_week",
                  "actual_net_revenue_week"]].head(8))


## 4. อ่านผล Test โดยไม่แต่งค่าให้ผ่าน

WAPE คือผลรวมความคลาดเคลื่อนสัมบูรณ์เทียบยอดจริง ยิ่งต่ำยิ่งดี เกณฑ์รายกลุ่ม ≤20% และทั้งร้าน ≤15% เป็น **เป้าหมายของโครงการ** ไม่ใช่มาตรฐานสากลหรือการรับประกันความแม่นยำ


In [ ]:
#@title 4 — เปรียบเทียบโมเดลกับยอดจริง
selection = pd.read_csv(OUT / "weekly_model_selection.csv")
scores = pd.read_csv(OUT / "weekly_test_metrics.csv")
predictions = pd.read_csv(OUT / "weekly_test_predictions.csv")
print("เลือกโมเดลจาก Validation:")
display(selection)
print("ผล Locked Test (ไม่ใช้เลือกโมเดล):")
display(scores[["model", "group_wape_pct", "shop_wape_pct",
                "group_target_20pct", "shop_target_15pct"]])
chosen = summary["selected_by_validation"]
print("ตัวอย่างยอดจริงเทียบยอดทำนายของ", chosen)
display(predictions.loc[predictions["model"].eq(chosen),
    ["forecast_week_start", "product_group_id", "actual_net_revenue_week",
     "predicted_net_revenue_week"]].head(10))


## 5. โหลดโมเดลที่บันทึกไว้และตรวจผลภายหลัง

ขั้นนี้เป็น **offline replay**: โหลดโมเดลที่บันทึกไว้เพื่อทำนายจาก Feature แล้วจึงเทียบยอดจริงสำหรับ monitoring ไม่ใช่ API ทำนายแบบสด และไม่พิสูจน์ว่าเพิ่ม Ads แล้วขายเพิ่ม


In [ ]:
#@title 5 — Serving replay และ monitoring
from pipeline.ml_weekly_serve import serve_monitor_replay
monitor = serve_monitor_replay(OUT)
served = pd.read_csv(OUT / "serving_replay_predictions.csv")
print("สถานะ:", monitor["production_approval"])
print("สัปดาห์ที่ Replay:", monitor["forecast_week"])
print("Alerts:", monitor["alerts"])
display(served)
print("ไฟล์ผลทั้งหมด:", OUT)

from pipeline.ml_scenarios import build_scenarios
scenarios = build_scenarios(WORKSPACE / "er2.duckdb", OUT)
print("Scenario:", scenarios["status"], "| ตัวเลือก:", len(scenarios.get("grid", [])))
if scenarios["status"] == "READY":
    display(pd.DataFrame(scenarios["grid"]).head(10))
print("การปรับงบเป็นการลองเปลี่ยน Feature ไม่ใช่ผลเพิ่มยอดขายเชิงเหตุและผล")
